# Coding Task 3 – Structured Prediction (New Notebook)

This notebook is a clean workspace for implementing and experimenting with:

- Structural SVM
- Conditional Random Fields (CRF)
- Auto-context
- Fixed-point inference

on the OCR dataset, with word-level sequences and sliding-window features.

## W&B logging (optional)

If you want to store results in Weights & Biases:

```bash
pip install wandb
wandb login
```

Then set `WANDB_ENABLED = True` in the imports cell (it is enabled by default but will auto-disable if `wandb` is not installed).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings; warnings.simplefilter('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

%matplotlib inline

from ocr_utils import (
    read_OCR,
    build_word_sequences,
    make_sliding_window_features,
    train_test_split_sequences,
)

from structural_svm import run_structured_svm
from crf_model import run_crf
from auto_context import run_auto_context
from fixed_point import run_fixed_point

# Optional experiment tracking
WANDB_ENABLED = True
WANDB_PROJECT = "ocr-comparison"
WANDB_RUN_NAME = None  # set to a string if you want a custom name

try:
    import wandb
except Exception as e:
    WANDB_ENABLED = False
    wandb = None
    print("wandb not available; skipping logging. Install with: pip install wandb")

if WANDB_ENABLED and wandb.run is None:
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        config={
            "N_WORDS": 5000,
            "D": 128,
            "K": 26,
        },
    )

In [3]:
# Parameters
N_WORDS = 5000   # total words to consider
D = 128          # per-character feature length
K = 26           # number of labels (a-z)

if WANDB_ENABLED:
    wandb.config.update({"N_WORDS": N_WORDS, "D": D, "K": K}, allow_val_change=True)

In [4]:
# Load dataset

dataset1 = read_OCR('OCRdataset/letter.data', D)

print("max label=", max(dataset1['labels']), "min label=", min(dataset1['labels']), "#labels=", len(dataset1['labelDic']))
print("Total letters=", len(dataset1['ids']))
print("Features shape=", np.array(dataset1['features']).shape)

max label= 25 min label= 0 #labels= 26
Total letters= 52152
Features shape= (52152, 128)


In [5]:
# Build word-level sequences once
word_features, word_labels = build_word_sequences(dataset1, max_words=N_WORDS, shuffle=True, random_state=0)
print("Number of words:", len(word_features))

Number of words: 5000


In [6]:
# === Smoke tests (tiny data) for each classifier ===
# Run this cell first to verify everything works end-to-end.

SMOKE_WINDOW_RADIUS = 1   # window size = 3
SMOKE_TRAIN = 50
SMOKE_TEST = 50

X_smoke = make_sliding_window_features(word_features, SMOKE_WINDOW_RADIUS)
d_window_smoke = X_smoke[0].shape[1]
print("Feature size: ", d_window_smoke)
X_tr_s, y_tr_s, X_te_s, y_te_s = train_test_split_sequences(
    X_smoke, word_labels, SMOKE_TRAIN, SMOKE_TEST, random_state=0
)

print("split data")

smoke_rows = []

def _record(name, res):
    smoke_rows.append({
        'method': name,
        'train_word_error': 1.0 - res['train_word_acc'],
        'test_word_error': 1.0 - res['test_word_acc'],
        'train_char_error': 1.0 - res['train_char_acc'],
        'test_char_error': 1.0 - res['test_char_acc'],
        'train_time': res['train_time'],
        'test_time': res['test_time'],
    })

# Structural SVM
res_ssvm_smoke = run_structured_svm(X_tr_s, y_tr_s, X_te_s, y_te_s, K=K, d_window=d_window_smoke)
_record('struct_svm', res_ssvm_smoke)
print(res_ssvm_smoke)

# CRF
res_crf_smoke = run_crf(X_tr_s, y_tr_s, X_te_s, y_te_s)
_record('crf', res_crf_smoke)
print(res_crf_smoke)

# Auto-context
res_auto_smoke = run_auto_context(X_tr_s, y_tr_s, X_te_s, y_te_s, K=K)
_record('auto_context', res_auto_smoke)
print(res_auto_smoke)

# Fixed-point
res_fp_smoke = run_fixed_point(X_tr_s, y_tr_s, X_te_s, y_te_s, K=K)
_record('fixed_point', res_fp_smoke)
print(res_fp_smoke)
smoke_df = pd.DataFrame(smoke_rows).sort_values('method')

if WANDB_ENABLED:
    wandb.log({"smoke/results": wandb.Table(dataframe=smoke_df)})

smoke_df

Feature size:  384
split data
{'weights': dlib.vector([-0, 0.0246359, 0.00279438, -0.0454385, 0.0454339, 0.0368123, -0.00358865, 0.0167004, 0.000281348, 0.0328023, -0.0548525, -0.0198376, 0.0589383, -0.0112976, -0.0252192, -0.0220044, 0.019212, 0.0186513, -0.0270249, 0.0124712, 0.0282986, -0.0312451, 0.00470218, -0.00133896, -0.0115059, -0.00272043, -0.00913315, -0.0495993, 0.0062916, -0.0115332, 0.00525387, -0.00621321, -0.00443323, -0.0478899, -0.0315994, -0.00239377, 0.00895247, -0.0608677, 0.0128964, 0.00893017, -0.0236299, -0.0303462, -0.0354593, 0.0170036, -0.042358, -0.0394559, 0.0294773, 0.021059, 0.0176099, -0.0259551, 0.00876669, 0.0317989, -0.0268684, -0.0301903, -0.0172455, 0.0201178, 0.00746192, 0.00563892, 0.053969, -0.051896, 0.0574095, 0.0438736, -0.0400364, -0.00404753, -0.0200605, 0.0412428, -0.0129516, 0.00506601, 0.0537024, -0.0274317, -0.0700866, -0.052383, 0.00934198, 0.0861877, 0.00343625, -0.0361155, -0.00424579, -0.0261102, -0.0625761, -0.0210229, 0.0224113, -0

,method,train_word_error,test_word_error,train_char_error,test_char_error,train_time,test_time
2,auto_context,0.0,0.90,0.000000,0.480519,2.653452,0.001377
1,crf,0.0,0.92,0.000000,0.498701,0.711689,0.034248
3,fixed_point,0.0,0.86,0.000000,0.459740,1.102025,0.014233
0,struct_svm,0.1,0.92,0.013228,0.483117,138.856608,0.471150


In [ ]:
# === Experiments: compare methods, window sizes, and train/test splits ===

window_radii = [0, 1, 2]  # window sizes 1, 3, 5
splits = [(1000, 4000), (2500, 2500), (4000, 1000)]

results = []
wandb_step = 0


def maybe_wandb_log(row):
    global wandb_step
    if WANDB_ENABLED:
        wandb.log(
            {
                "method": row["method"],
                "window_radius": row["window_radius"],
                "window_size": row["window_size"],
                "n_train": row["n_train"],
                "n_test": row["n_test"],
                "train_word_error": row["train_word_error"],
                "test_word_error": row["test_word_error"],
                "train_char_error": row["train_char_error"],
                "test_char_error": row["test_char_error"],
                "train_time": row["train_time"],
                "test_time": row["test_time"],
            },
            step=wandb_step,
        )
    wandb_step += 1


for W in window_radii:
    print("==== Window radius {} (window size {}) ====".format(W, 2 * W + 1))
    X_win = make_sliding_window_features(word_features, W)
    d_window = X_win[0].shape[1]

    for n_train, n_test in splits:
        print("-- Split train/test = {}/{}".format(n_train, n_test))
        X_tr, y_tr, X_te, y_te = train_test_split_sequences(X_win, word_labels, n_train, n_test, random_state=0)

        res_ssvm = run_structured_svm(X_tr, y_tr, X_te, y_te, K=K, d_window=d_window)
        row = {
            'method': 'struct_svm',
            'window_radius': W,
            'window_size': 2 * W + 1,
            'n_train': n_train,
            'n_test': n_test,
            'train_word_error': 1.0 - res_ssvm['train_word_acc'],
            'test_word_error': 1.0 - res_ssvm['test_word_acc'],
            'train_char_error': 1.0 - res_ssvm['train_char_acc'],
            'test_char_error': 1.0 - res_ssvm['test_char_acc'],
            'train_time': res_ssvm['train_time'],
            'test_time': res_ssvm['test_time'],
        }
        results.append(row)
        maybe_wandb_log(row)

        res_crf = run_crf(X_tr, y_tr, X_te, y_te)
        row = {
            'method': 'crf',
            'window_radius': W,
            'window_size': 2 * W + 1,
            'n_train': n_train,
            'n_test': n_test,
            'train_word_error': 1.0 - res_crf['train_word_acc'],
            'test_word_error': 1.0 - res_crf['test_word_acc'],
            'train_char_error': 1.0 - res_crf['train_char_acc'],
            'test_char_error': 1.0 - res_crf['test_char_acc'],
            'train_time': res_crf['train_time'],
            'test_time': res_crf['test_time'],
        }
        results.append(row)
        maybe_wandb_log(row)

        res_auto = run_auto_context(X_tr, y_tr, X_te, y_te, K=K)
        row = {
            'method': 'auto_context',
            'window_radius': W,
            'window_size': 2 * W + 1,
            'n_train': n_train,
            'n_test': n_test,
            'train_word_error': 1.0 - res_auto['train_word_acc'],
            'test_word_error': 1.0 - res_auto['test_word_acc'],
            'train_char_error': 1.0 - res_auto['train_char_acc'],
            'test_char_error': 1.0 - res_auto['test_char_acc'],
            'train_time': res_auto['train_time'],
            'test_time': res_auto['test_time'],
        }
        results.append(row)
        maybe_wandb_log(row)

        res_fp = run_fixed_point(X_tr, y_tr, X_te, y_te, K=K)
        row = {
            'method': 'fixed_point',
            'window_radius': W,
            'window_size': 2 * W + 1,
            'n_train': n_train,
            'n_test': n_test,
            'train_word_error': 1.0 - res_fp['train_word_acc'],
            'test_word_error': 1.0 - res_fp['test_word_acc'],
            'train_char_error': 1.0 - res_fp['train_char_acc'],
            'test_char_error': 1.0 - res_fp['test_char_acc'],
            'train_time': res_fp['train_time'],
            'test_time': res_fp['test_time'],
        }
        results.append(row)
        maybe_wandb_log(row)

results_df = pd.DataFrame(results).sort_values(['method', 'window_radius', 'n_train'])

if WANDB_ENABLED:
    wandb.log({"results/table": wandb.Table(dataframe=results_df)})

results_df

In [ ]:
# Optional: quick pivots for comparison

# Mean errors/times grouped by method/window/split
summary = results_df.copy()
summary['split'] = summary['n_train'].astype(str) + '/' + summary['n_test'].astype(str)

pivot_err = summary.pivot_table(
    index=['method', 'window_size'],
    columns='split',
    values=['test_word_error', 'test_char_error', 'train_time', 'test_time'],
    aggfunc='mean'
)

pivot_err